# LLM Paper Reviewer

Upload a PDF and get an automated peer review from an LLM (Gemini or OpenAI).

**Setup:** Set your API key in the environment before running:
```bash
export GEMINI_API_KEY="your-key"   # for Gemini
export OPENAI_API_KEY="sk-..."     # for OpenAI
```

In [1]:
# ── Config ──────────────────────────────────────────────
PDF_PATH = "data/1253_diffusion_based_voice_conversi.pdf"  # <-- change this
PROVIDER = "gemini"  # "gemini" or "openai"
MODEL = None  # None = provider default (gemini-2.0-flash / gpt-4o-mini)
TEMPERATURE = 0.3
SAVE_REVIEW = True  # save JSON review next to the PDF

In [2]:
# ── Imports & helpers ───────────────────────────────────
import os, json, re, pathlib
from dotenv import load_dotenv

load_dotenv()  # loads .env into os.environ

def parse_json_response(raw_text: str) -> dict:
    """Extract and parse a JSON object from an LLM response."""
    text = raw_text.strip()
    # strip markdown fences
    m = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
    if m:
        text = m.group(1)
    elif text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # attempt to close a truncated object
        try:
            return json.loads(text.rstrip(',"') + '}')
        except json.JSONDecodeError:
            return {"error": "Failed to parse JSON", "raw": text[:500]}

REVIEW_PROMPT = (
    "You are an expert peer reviewer for top-tier CS venues.\n"
    "Write a concise, critical review as valid JSON:\n"
    "{\n"
    '  \"summary\": \"2-3 sentence summary\",\n'
    '  \"strengths\": [\"3-4 key strengths, 1 sentence each\"],\n'
    '  \"weaknesses\": [\"3-4 key weaknesses, 1 sentence each\"],\n'
    '  \"questions\": [\"2-3 specific questions\"],\n'
    '  \"score\": {\"value\": 5, \"justification\": \"Brief reason\"},\n'
    '  \"decision\": \"Accept/Weak Accept/Borderline/Weak Reject/Reject\"\n'
    "}\n"
    "Keep each field concise. Ensure valid JSON syntax.\n"
)

print("Helpers loaded.")

Helpers loaded.


In [3]:
# ── Build the LLM client ────────────────────────────────
import time

if PROVIDER == "gemini":
    import google.genai as genai

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    model_name = MODEL or "gemini-2.5-flash"

    def review_pdf(pdf_path: str, max_retries: int = 3) -> dict:
        """Upload a PDF directly to Gemini and return a parsed review."""
        uploaded = client.files.upload(file=pdf_path)
        print(uploaded)
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        REVIEW_PROMPT + "\n\nReview the attached paper.",
                        uploaded,
                    ],
                    
                )
                return parse_json_response(response.text)
            except Exception as e:
                print(e)
                if "429" in str(e) and attempt < max_retries - 1:
                    wait = 25 * (attempt + 1)
                    print(f"Rate limited, retrying in {wait}s ...", end=" ", flush=True)
                    time.sleep(wait)
                else:
                    print(f"LLM error: {repr(e)}")
                    raise


else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")

print(f"Using {PROVIDER} / {model_name}")

Using gemini / gemini-2.5-flash


In [ ]:
# ── Generate the review ─────────────────────────────────
# Fetch the PDF from OpenReview and review using Gemini
import requests
import os

# Ensure the data and reviews directories exist
os.makedirs("data", exist_ok=True)
os.makedirs("reviews", exist_ok=True)

# List of ICLR 2026 papers to fetch and review (sample from Accept/Reject tabs)
PAPERS_TO_REVIEW = [
    {
        "title": "Mixture-of-Experts Can Surpass Dense LLMs Under Strictly Equal Compute",
        "pdf_url": "https://openreview.net/attachment?id=odjMSBSWRt&name=pdf",
        "category": "accept"
    },
    {
        "title": "Improving Developer Emotion Classification via LLM-Based Augmentation",
        "pdf_url": "https://openreview.net/attachment?id=FPLNSx1jmL&name=pdf",
        "category": "reject"
    },
]

for paper in PAPERS_TO_REVIEW:
    pdf_filename = f"data/{paper['title'].replace(' ', '_').replace(':', '').replace('/', '_')}.pdf"
    review_filename = f"reviews/{paper['title'].replace(' ', '_').replace(':', '').replace('/', '_')}.review.json"
    # Download PDF
    try:
        r = requests.get(paper['pdf_url'])
        r.raise_for_status()
        with open(pdf_filename, "wb") as f:
            f.write(r.content)
        print(f"Downloaded PDF: {pdf_filename}")
    except Exception as e:
        print(f"Failed to download PDF: {e}")
        continue
    # Review with Gemini
    print(f"Reviewing: {pdf_filename} ...")
    review = review_pdf(pdf_filename)
    print(json.dumps(review, indent=2))
    # Save review in reviews folder
    with open(review_filename, "w") as f:
        f.write(json.dumps(review, indent=2))
    print(f"Review saved → {review_filename}")

Downloaded PDF: data/Mixture-of-Experts_Can_Surpass_Dense_LLMs_Under_Strictly_Equal_Compute.pdf
Reviewing: data/Mixture-of-Experts_Can_Surpass_Dense_LLMs_Under_Strictly_Equal_Compute.pdf ...
name='files/p908y44shfsp' display_name=None mime_type='application/pdf' size_bytes=618180 create_time=datetime.datetime(2026, 2, 13, 19, 14, 53, 1560, tzinfo=TzInfo(0)) expiration_time=datetime.datetime(2026, 2, 15, 19, 14, 52, 831345, tzinfo=TzInfo(0)) update_time=datetime.datetime(2026, 2, 13, 19, 14, 53, 1560, tzinfo=TzInfo(0)) sha256_hash='ZTAzNDg0YTNiNzUwYWU1MDQ3YjA1MjNkNTdhZTExZWZjM2JjMzczZWNkOWVjMGZiMDJlNjgyNDZiNGIxYjZjYQ==' uri='https://generativelanguage.googleapis.com/v1beta/files/p908y44shfsp' download_uri=None state=<FileState.ACTIVE: 'ACTIVE'> source=<FileSource.UPLOADED: 'UPLOADED'> video_metadata=None error=None
{
  "summary": "This paper introduces DarkBench, a novel benchmark to measure the prevalence of six dark design patterns in Large Language Models (LLMs). The authors evaluate

In [5]:
# ── (Optional) Save the review ──────────────────────────
if SAVE_REVIEW and 'review' in locals() and review is not None:
    out_path = pathlib.Path(PDF_PATH).with_suffix(".review.json")
    out_path.write_text(json.dumps(review, indent=2))
    print(f"Review saved → {out_path}")
else:
    print("No review to save.")

Review saved → data/1253_diffusion_based_voice_conversi.review.json


---
## Batch mode
Point `PDF_DIR` at a folder of PDFs to review them all.

In [6]:
# ── Batch review for ICLR 2026 sampled papers ──
import requests
from PyPDF2 import PdfReader

# Sampled papers from Accept (Oral) and Reject
PAPERS = [
    {
        "title": "Common Corpus: The Largest Collection of Ethical Data for LLM Pre-Training",
        "pdf_url": "https://openreview.net/attachment?id=0wSlFpMsGb&name=pdf",
        "category": "accept"
    },
    {
        "title": "Q-RAG: Long Context Multi‑Step Retrieval via Value‑Based Embedder Training",
        "pdf_url": "https://openreview.net/attachment?id=MS9nWFY7LG&name=pdf",
        "category": "accept"
    },
    {
        "title": "Improving Developer Emotion Classification via LLM-Based Augmentation",
        "pdf_url": "https://openreview.net/attachment?id=FPLNSx1jmL&name=pdf",
        "category": "reject"
    },
    {
        "title": "Revisiting Multilingual Data Mixtures in Language Model Pretraining",
        "pdf_url": "https://openreview.net/attachment?id=IKJyRyHpHV&name=pdf",
        "category": "reject"
    },
    # Add more if needed
 ]
def download_pdf(url, out_path):
    r = requests.get(url)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path
def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text
results = []
for paper in PAPERS:
    print(f"\n--- Reviewing: {paper['title']} ({paper['category']}) ---")
    pdf_filename = f"data/{paper['title'].replace(' ', '_').replace(':', '').replace('/', '_')}.pdf"
    # Download PDF
    try:
        download_pdf(paper['pdf_url'], pdf_filename)
    except Exception as e:
        print(f"Failed to download PDF: {e}")
        results.append({"title": paper['title'], "error": f"Download failed: {e}"})
        continue
    # Extract PDF text
    try:
        pdf_text = extract_pdf_text(pdf_filename)
        print(f"PDF-to-text extraction: {len(pdf_text)} chars")
    except Exception as e:
        print(f"Failed to extract text: {e}")
        pdf_text = None
    # Submit PDF text to Gemini
    text_review = None
    if pdf_text:
        try:
            text_path = pdf_filename.replace('.pdf', '.txt')
            with open(text_path, 'w') as f:
                f.write(pdf_text)
            text_review = review_pdf(text_path)
            print(f"Text review score: {text_review.get('score', {}).get('value', '?')}")
        except Exception as e:
            print(f"Text review failed: {e}")
    # Submit PDF file to Gemini
    try:
        pdf_review = review_pdf(pdf_filename)
        print(f"PDF review score: {pdf_review.get('score', {}).get('value', '?')}")
    except Exception as e:
        print(f"PDF review failed: {e}")
        pdf_review = None
    # Consistency check
    consistency = None
    if text_review and pdf_review:
        consistency = {
            "score_match": text_review.get('score', {}).get('value') == pdf_review.get('score', {}).get('value'),
            "decision_match": text_review.get('decision') == pdf_review.get('decision')
        }
        print(f"Consistency: {consistency}")
    results.append({
        "title": paper['title'],
        "category": paper['category'],
        "text_review": text_review,
        "pdf_review": pdf_review,
        "consistency": consistency
    })
print("\nBatch review results:")
for r in results:
    print(json.dumps(r, indent=2))


--- Reviewing: Common Corpus: The Largest Collection of Ethical Data for LLM Pre-Training (accept) ---
PDF-to-text extraction: 110442 chars
name='files/zhpo8ec67rof' display_name=None mime_type='text/plain' size_bytes=110814 create_time=datetime.datetime(2026, 2, 13, 19, 15, 24, 672826, tzinfo=TzInfo(0)) expiration_time=datetime.datetime(2026, 2, 15, 19, 15, 24, 215998, tzinfo=TzInfo(0)) update_time=datetime.datetime(2026, 2, 13, 19, 15, 24, 672826, tzinfo=TzInfo(0)) sha256_hash='NzMzZmY4NDlkMjZkMGYxYjhkYzQxN2ZlYTUxZWRmMDU1N2EzYTIwOTVlYTEwZjQwZTAxODMyNTg0YjVmZDA3NQ==' uri='https://generativelanguage.googleapis.com/v1beta/files/zhpo8ec67rof' download_uri=None state=<FileState.ACTIVE: 'ACTIVE'> source=<FileSource.UPLOADED: 'UPLOADED'> video_metadata=None error=None
Text review score: 6
name='files/c585dg4m42hp' display_name=None mime_type='application/pdf' size_bytes=765855 create_time=datetime.datetime(2026, 2, 13, 19, 15, 36, 326081, tzinfo=TzInfo(0)) expiration_time=datetime.datetime

In [7]:
# Single paper upload and Gemini review (ICLR 2026 example)
import requests
from PyPDF2 import PdfReader

# Choose a single paper (example: Accept Oral)
PAPER = {
    "title": "Common Corpus: The Largest Collection of Ethical Data for LLM Pre-Training",
    "pdf_url": "https://openreview.net/attachment?id=0wSlFpMsGb&name=pdf",
    "category": "accept"
}

pdf_filename = f"data/{PAPER['title'].replace(' ', '_').replace(':', '').replace('/', '_')}.pdf"

# Download PDF
try:
    r = requests.get(PAPER['pdf_url'])
    r.raise_for_status()
    with open(pdf_filename, "wb") as f:
        f.write(r.content)
    print(f"Downloaded PDF: {pdf_filename}")
except Exception as e:
    print(f"Failed to download PDF: {e}")

# Extract PDF text
try:
    reader = PdfReader(pdf_filename)
    pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    print(f"PDF-to-text extraction: {len(pdf_text)} chars")
except Exception as e:
    print(f"Failed to extract text: {e}")
    pdf_text = None

# Submit PDF file to Gemini
try:
    pdf_review = review_pdf(pdf_filename)
    print(f"PDF review score: {pdf_review.get('score', {}).get('value', '?')}")
    print(json.dumps(pdf_review, indent=2))
except Exception as e:
    print(f"PDF review failed: {e}")

# (Optional) Submit PDF text to Gemini for comparison
if pdf_text:
    try:
        text_path = pdf_filename.replace('.pdf', '.txt')
        with open(text_path, 'w') as f:
            f.write(pdf_text)
        text_review = review_pdf(text_path)
        print(f"Text review score: {text_review.get('score', {}).get('value', '?')}")
        print(json.dumps(text_review, indent=2))
    except Exception as e:
        print(f"Text review failed: {e}")

Downloaded PDF: data/Common_Corpus_The_Largest_Collection_of_Ethical_Data_for_LLM_Pre-Training.pdf
PDF-to-text extraction: 110442 chars
name='files/czr6dzc21msv' display_name=None mime_type='application/pdf' size_bytes=765855 create_time=datetime.datetime(2026, 2, 13, 19, 17, 9, 450885, tzinfo=TzInfo(0)) expiration_time=datetime.datetime(2026, 2, 15, 19, 17, 9, 151602, tzinfo=TzInfo(0)) update_time=datetime.datetime(2026, 2, 13, 19, 17, 9, 450885, tzinfo=TzInfo(0)) sha256_hash='ZmRkOWZlMjdlOTg3NzE5YzljNDUyY2RlZDZiYWJkOTRhMzcxNTU1ZDk3NTZmYTNhNGNmZDBmZDNiZmMxNjIyMQ==' uri='https://generativelanguage.googleapis.com/v1beta/files/czr6dzc21msv' download_uri=None state=<FileState.ACTIVE: 'ACTIVE'> source=<FileSource.UPLOADED: 'UPLOADED'> video_metadata=None error=None
PDF review score: ?
{
  "error": "Failed to parse JSON",
  "raw": "{\n  \"summary\": \"This paper introduces Common Corpus, a novel 2-trillion-token multilingual dataset meticulously curated for ethical compliance and open licen